# Tree Species Classification with Transfer Learning

**Course:** Neural Networks for Computer Vision — HAMK 2026

**Group members:**
- Timofey Krylov (timofey.krylov@student.hamk.fi)
- Tomi Kitola (tomi.kitola@student.hamk.fi)
- Jaakko Ruhanen (jaakko.ruhanen@student.hamk.fi)

## Project overview

We classify **23 urban street tree species** from phone photos with pretrained convolutional neural networks.
Two ResNet architectures of different size — **ResNet18** (~11 M parameters) and **ResNet50** (~24 M parameters) —
are fine-tuned on the same data, following the PyTorch
[transfer learning tutorial](https://docs.pytorch.org/tutorials/beginner/transfer_learning_tutorial.html).

**Dataset:** [Tree Dataset of Urban Street — Classification Tree](https://www.kaggle.com/datasets/erickendric/tree-dataset-of-urban-street-classification-tree)

**Notebook structure**
1. Dataset exploration and preprocessing (dataset's own train / validation / test split)
2. Transforms, data augmentation, DataLoaders and the two ResNet models
3. **Training round 1** — without augmentation and with the tutorial augmentation
4. **Testing** — accuracy, precision, recall, F1, confusion matrices, n-best (top-n) accuracy
5. **Error analysis** — systematic errors and data problems found in the images
6. **Training round 2** — improved models (data fixes based on the error analysis)
7. Final comparison and conclusions

### How to run (Google Colab)
1. `Runtime → Change runtime type → T4 GPU`
2. `Runtime → Run all`. Google Drive is mounted for saving the trained models.
3. Trained weights are saved to `MyDrive/nncv_tree_models_final/`. If a model already exists there, it is
   **loaded instead of trained again** (set `FORCE_RETRAIN = True` to retrain).

The notebook also runs locally (Jupyter) — then models are saved to a local `models/` folder.
Six models are trained in total; on a Colab T4 a full run from scratch takes roughly 45–60 minutes.

## 0. Environment setup

In [ ]:
# kagglehub is used to download the dataset (preinstalled in Colab, installed here if missing)
%pip install -q kagglehub

In [ ]:
import os, random, time, copy, json, hashlib
import numpy as np
import torch, torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler
from torch.utils.data import DataLoader
import torchvision
from torchvision import models, transforms as T
from PIL import Image, ImageOps
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from sklearn.metrics import (confusion_matrix, accuracy_score, precision_recall_fscore_support,
                             classification_report)

plt.rcParams["figure.dpi"] = 90

# Reproducibility
SEED = 72
torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    torch.backends.cudnn.benchmark = True        # fastest conv algorithms for a fixed input size
    torch.set_float32_matmul_precision("high")   # TF32 for the remaining fp32 math (Ampere+)
    # Mixed precision: bf16 on Ampere and newer (RTX 30/40, A100), fp16 + gradient scaling on older GPUs (Colab T4)
    AMP_DTYPE = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16
    print("GPU:", torch.cuda.get_device_name(0), "| mixed precision:", AMP_DTYPE)
else:
    AMP_DTYPE = torch.bfloat16
    print("No GPU found — training will be very slow (Colab: Runtime → Change runtime type → T4 GPU)")
USE_AMP = device.type == "cuda"

NUM_WORKERS = min(8, os.cpu_count() or 2)   # parallel image loading (Colab has 2 CPU cores)

In [ ]:
# Trained models are saved outside the submission (too big for the zip).
# Colab -> Google Drive, local Jupyter -> "models" folder
try:
    from google.colab import drive
    drive.mount("/content/drive")
    SAVE_DIR = "/content/drive/MyDrive/nncv_tree_models_final"
except ImportError:
    SAVE_DIR = "models"
os.makedirs(SAVE_DIR, exist_ok=True)

FORCE_RETRAIN = False   # False: load a model from SAVE_DIR if it was already trained, True: always train again
print("Models are saved to:", SAVE_DIR)

In [ ]:
%%writefile tree_data.py
# The Dataset class lives in its own .py file so that DataLoader worker processes can import it
# (needed on Windows / macOS; on Linux and Colab it would also work inside the notebook).
from PIL import Image
from torch.utils.data import Dataset

class TreeDataset(Dataset):
    # Simple dataset: list of image paths + list of labels
    def __init__(self, paths, labels, transform):
        self.paths, self.labels, self.transform = paths, labels, transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert("RGB")
        return self.transform(img), self.labels[i]

In [ ]:
from tree_data import TreeDataset

## 1. Dataset: Tree Dataset of Urban Street

- Photos of street trees taken with mobile phones in 10 cities in China, in different seasons.
- One folder per species → the folder name is the class label.
- The dataset comes with its **own stratified split into training (80 %), validation (10 %) and test (10 %)**
  sets (`tree/train`, `tree/val`, `tree/test`). We use this split as is, so our results are comparable with other
  work on the same dataset. The test set is used only for the final evaluation.

In [ ]:
# Download the dataset from Kaggle (cached after the first time)
import kagglehub

path = kagglehub.dataset_download("erickendric/tree-dataset-of-urban-street-classification-tree")
print("Dataset folder:", path)

In [ ]:
# Inspect the dataset folder structure (3 levels)
for root, dirs, files in os.walk(path):
    depth = root[len(path):].count(os.sep)
    if depth <= 2:
        print("    " * depth + os.path.basename(root) + f"/  ({len(files)} files)")

In [ ]:
# Collect every image: label = species folder name, split = train / val / test folder name
# Folder structure: tree/<split>/<species>/<file>
IMG_EXT = (".jpg", ".jpeg", ".png", ".bmp")

records = []
for root, dirs, files in os.walk(path):
    for f in files:
        if f.lower().endswith(IMG_EXT):
            records.append(dict(path=os.path.join(root, f),
                                species=os.path.basename(root),
                                split=os.path.basename(os.path.dirname(root))))

# Sorted -> same order on every computer (os.walk order depends on the file system)
df = pd.DataFrame(records).sort_values(["split", "species", "path"]).reset_index(drop=True)
class_names = sorted(df["species"].unique())
num_classes = len(class_names)
df["label"] = df["species"].map(class_names.index)

print("Images:", len(df), "| Classes:", num_classes)
print(class_names)
df["split"].value_counts()

In [ ]:
# Number of images per species and split
counts = pd.crosstab(df["species"], df["split"])[["train", "val", "test"]]

counts.plot.barh(stacked=True, figsize=(8, 7))
plt.xlabel("Number of images"); plt.title("Images per species"); plt.gca().invert_yaxis()
plt.tight_layout(); plt.show()

total = counts.sum(axis=1)
print("Smallest class:", total.idxmin(), total.min(), "| Biggest class:", total.idxmax(), total.max())
print("Share of each split per species (min–max): " +
      ", ".join(f"{s} {(counts[s] / total).min():.0%}–{(counts[s] / total).max():.0%}" for s in counts))

The classes are imbalanced (81 – 292 images per species), so besides accuracy we report **macro-averaged**
precision, recall and F1, where every species counts the same. The provided split is stratified: every species
is split ≈ 80 / 10 / 10.

In [ ]:
# Random samples from the dataset (shown as they are stored in the files)
fig, axs = plt.subplots(3, 4, figsize=(12, 8))
for ax, i in zip(axs.ravel(), random.sample(range(len(df)), 12)):
    ax.imshow(ImageOps.exif_transpose(Image.open(df.path[i])))
    ax.set_title(df.species[i], fontsize=9); ax.axis("off")
plt.tight_layout(); plt.show()

## 2. Pixels & image tensors

- The tree photos are **RGB** images: PIL / NumPy format `(H, W, 3)`, PyTorch tensor format `(3, H, W)`.
- They are large phone photos (1600 × 1200), much bigger than e.g. MNIST (28 × 28). Image size has a big effect on
  memory use and training speed.

In [ ]:
# Inspect one tree image
img = ImageOps.exif_transpose(Image.open(df.path[0])).convert("RGB")
x = T.ToTensor()(img)
print("PIL size (W, H):", img.size)
print("Tensor shape:", x.shape, "| dtype:", x.dtype, "| min/max:", float(x.min()), float(x.max()))
print("Flattened 224x224 RGB input:", 3 * 224 * 224, "values (MNIST:", 28 * 28, ")")

# Sizes of all original photos (PIL reads only the header -> fast)
sizes = pd.Series([Image.open(p).size for p in df.path]).value_counts()
print("\nMost common sizes (W, H):")
print(sizes.head(5).to_string())

## 3. What is image classification here?

- Input: RGB image tensor `[3, 224, 224]` → output: one of **23 tree species**.
- The model outputs **23 logits**, one per class; training uses **CrossEntropyLoss**.
- Evaluation: **accuracy, precision, recall, F1** (macro average), **confusion matrix** and **n-best (top-n) accuracy**.

## 4. Image preprocessing

The original photos are 1600 × 1200 px. Decoding them every epoch would be much slower than the training itself,
and the network only sees 224 × 224 crops. Therefore every photo is resized **once** so that its shorter side is
256 px (aspect ratio kept, same as `Resize(256)` in the tutorial) and saved as a small JPEG.

- `ImageOps.exif_transpose` applies a possible EXIF rotation tag, and images are converted to RGB.
- JPEG *draft mode* decodes the photo directly at a reduced size → preprocessing is several times faster.

In [ ]:
shrink = T.Resize(256)   # shorter side -> 256 px, keeps the aspect ratio

def load_photo(p):
    # Open a photo at reduced size (JPEG draft mode: still >= 256 px on the shorter side) as RGB
    im = Image.open(p)
    im.draft("RGB", (256, 256))
    return ImageOps.exif_transpose(im).convert("RGB")


def preprocess(out_dir):
    # Resize all photos once (shorter side 256 px) and save them to out_dir
    os.makedirs(out_dir, exist_ok=True)
    out_paths, start = [], time.time()
    for i, p in enumerate(df.path):
        out_path = os.path.join(out_dir, f"{i}.jpg")
        if not os.path.exists(out_path):
            shrink(load_photo(p)).save(out_path, quality=95)
        out_paths.append(out_path)
    print(f"{len(out_paths)} images ready in {out_dir}/ ({time.time() - start:.0f} s)")
    return out_paths


df["small"] = preprocess("data_small")


def pick(paths_col, split):
    # Paths and labels of one split
    rows = df[df["split"] == split]
    return list(rows[paths_col]), list(rows["label"])

splits = {s: pick("small", s) for s in ["train", "val", "test"]}
for s, (p, y) in splits.items():
    print(f"{s:5s}: {len(p)} images")

## 5. Transforms & augmentation

**Why transforms?**
- Put images into the right format and range (`ToTensor`).
- Standardize every color channel (`Normalize`) with the **ImageNet mean / std**, because the pretrained ResNet
  weights were trained with those values.
- Make all images the same size for batching (`Resize` / `CenterCrop` / `RandomResizedCrop`).
- Increase the diversity of the training data with random augmentations → less overfitting.

In [ ]:
img = Image.open(splits["train"][0][0]).convert("RGB")

# Resizing: direct resize distorts the tree, resize + center crop keeps the aspect ratio
variants = [("Original (preprocessed)", img),
            ("Resize((224, 224))", T.Resize((224, 224))(img)),
            ("Resize(256) + CenterCrop(224)", T.Compose([T.Resize(256), T.CenterCrop(224)])(img))]
fig, axs = plt.subplots(1, 3, figsize=(12, 4))
for ax, (name, im) in zip(axs, variants):
    ax.imshow(im); ax.set_title(name, fontsize=10); ax.axis("off")
plt.tight_layout(); plt.show()

# Typical data augmentations
augs = [("Rotate 15°", T.RandomRotation(15)),
        ("Horizontal flip", T.RandomHorizontalFlip(p=1.0)),
        ("Color jitter", T.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.1)),
        ("Perspective", T.RandomPerspective(distortion_scale=0.4, p=1.0)),
        ("Gaussian blur", T.GaussianBlur(kernel_size=9, sigma=(0.1, 2.0))),
        ("Grayscale", T.RandomGrayscale(p=1.0)),
        ("RandomResizedCrop", T.RandomResizedCrop(224))]
fig, axs = plt.subplots(2, 4, figsize=(12, 6))
axs = axs.ravel()
axs[0].imshow(img); axs[0].set_title("Original", fontsize=10); axs[0].axis("off")
for ax, (name, aug) in zip(axs[1:], augs):
    ax.imshow(aug(img)); ax.set_title(name, fontsize=10); ax.axis("off")
plt.tight_layout(); plt.show()

### Transform recipes — round 1

- **Training (augmented):** `RandomResizedCrop(224)` + `RandomHorizontalFlip` — the recipe of the transfer learning tutorial.
- **Training (no augmentation):** the same deterministic transform as validation / test — used to measure what
  augmentation actually brings.
- **Validation and test:** `Resize(256)` + `CenterCrop(224)` — no randomness, identical in every experiment.
- All images are converted to tensors and normalized with the ImageNet values.

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_tfms = T.Compose([
    T.RandomResizedCrop(224),          # random part of the image (8–100 % of the area), resized to 224x224
    T.RandomHorizontalFlip(),          # mirror image
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

val_tfms = T.Compose([
    T.Resize(256),
    T.CenterCrop(224),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
train_tfms, val_tfms

### Dataset & DataLoaders

In [ ]:
def make_loaders(train_transform, splits, batch_size=32):
    # val / test always use val_tfms -> only the training data / transform changes between experiments
    def loader(split, tfms, bs, shuffle):
        p, y = splits[split]
        return DataLoader(TreeDataset(p, y, tfms), batch_size=bs, shuffle=shuffle, num_workers=NUM_WORKERS,
                          pin_memory=device.type == "cuda", persistent_workers=NUM_WORKERS > 0)
    return {"train": loader("train", train_transform, batch_size, True),
            "val":   loader("val", val_tfms, 128, False),
            "test":  loader("test", val_tfms, 128, False)}


loaders = make_loaders(train_tfms, splits)

xb, yb = next(iter(loaders["train"]))
print(xb.shape, yb.shape)   # expected: [32, 3, 224, 224], [32]

## 6. Modeling: pretrained ResNets (transfer learning)

| | **ResNet18** | **ResNet50** |
|---|---|---|
| Depth | 18 layers | 50 layers |
| Block type | BasicBlock (2 × 3×3 conv) | Bottleneck (1×1 → 3×3 → 1×1 conv) |
| Parameters | ~11.7 M | ~25.6 M |
| Features before `fc` | 512 | 2048 |
| ImageNet top-1 accuracy | 69.8 % | 76.1 % |

**Transfer learning (fine-tuning)**
- Load weights pretrained on ImageNet (1000 classes) → the convolution layers already detect edges, textures,
  leaves, bark…
- Replace the last layer `fc` with a new `Linear(features → 23)`.
- Train **all** layers with a small learning rate ("finetuning the ConvNet" option of the tutorial).

In [ ]:
def make_model(name, num_classes=num_classes):
    # Pretrained ResNet with a new output layer for our 23 classes
    if name == "resnet18":
        model = models.resnet18(weights="IMAGENET1K_V1")
    elif name == "resnet50":
        model = models.resnet50(weights="IMAGENET1K_V1")
    else:
        raise ValueError(name)
    model.fc = nn.Linear(model.fc.in_features, num_classes)   # 512 -> 23 or 2048 -> 23
    return model.to(device, memory_format=torch.channels_last)


def count_params(model):
    return sum(p.numel() for p in model.parameters())


for name in ["resnet18", "resnet50"]:
    m = make_model(name)
    print(f"{name}: {count_params(m) / 1e6:.1f} M parameters | last layer: {m.fc}")
del m

### Training & evaluation helpers

- `train_epochs` trains with **CrossEntropyLoss**, **Adam** and the **StepLR** scheduler of the tutorial
  (learning rate × 0.1 every `step_size` epochs), tracks training and validation loss / accuracy, and returns the
  weights of the epoch with the **best validation accuracy**. The test set is never used here.
- Mixed precision (`torch.autocast`) makes training ~2× faster on the GPU.
- `fit_or_load` saves every trained model (+ its training history) to `SAVE_DIR`, and loads it from there on
  later runs instead of training again.
- Evaluation helpers: accuracy, macro precision / recall / F1, confusion matrices, most common errors,
  misclassified images and top-n accuracy.

In [ ]:
# ---------------- Training helpers ----------------

def denorm(x, mean=IMAGENET_MEAN, std=IMAGENET_STD):
    # Undo ImageNet normalization for display; x has shape [H, W, 3]
    return (x * torch.tensor(std) + torch.tensor(mean)).clamp(0, 1)


def show_batch(loader, n=16, nrow=8, title="Sample training batch"):
    # Show images from a DataLoader (after transforms / augmentation)
    xb, yb = next(iter(loader))
    grid = torchvision.utils.make_grid(xb[:n], nrow=nrow, padding=2)
    plt.figure(figsize=(14, 4))
    plt.imshow(denorm(grid.permute(1, 2, 0)))
    plt.axis("off"); plt.title(title); plt.show()
    print("Labels:", [class_names[i] for i in yb[:n]])


def to_device(xb, yb):
    return (xb.to(device, non_blocking=True, memory_format=torch.channels_last),
            yb.to(device, non_blocking=True))


def evaluate_loader(model, loader, loss_fn):
    # Average loss and accuracy over a loader
    model.eval()
    running, correct = 0.0, 0
    with torch.inference_mode(), torch.autocast(device.type, dtype=AMP_DTYPE, enabled=USE_AMP):
        for xb, yb in loader:
            xb, yb = to_device(xb, yb)
            out = model(xb)
            running += loss_fn(out.float(), yb).item() * xb.size(0)
            correct += (out.argmax(dim=1) == yb).sum().item()
    n = len(loader.dataset)
    return running / n, correct / n


def train_epochs(model, train_loader, val_loader, epochs=15, lr=1e-4, wd=1e-4, step_size=7, label_smoothing=0.0):
    # Train a model for several epochs with validation after each epoch.
    # Returns the model with the BEST validation weights + the history dictionary.
    ce = nn.CrossEntropyLoss(label_smoothing=label_smoothing)
    opt = optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    sched = lr_scheduler.StepLR(opt, step_size=step_size, gamma=0.1)   # from the transfer learning tutorial
    scaler = torch.amp.GradScaler(device.type, enabled=USE_AMP and AMP_DTYPE == torch.float16)
    hist = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    best_acc, best_wts = 0.0, copy.deepcopy(model.state_dict())
    start = time.time()

    for ep in range(1, epochs + 1):
        model.train()
        running = torch.zeros((), device=device)
        correct = torch.zeros((), device=device, dtype=torch.long)

        for xb, yb in train_loader:
            xb, yb = to_device(xb, yb)
            with torch.autocast(device.type, dtype=AMP_DTYPE, enabled=USE_AMP):
                out = model(xb)
                loss = ce(out, yb)

            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()

            running += loss.detach() * xb.size(0)   # stays on the GPU -> no sync every step
            correct += (out.argmax(1) == yb).sum()
        sched.step()

        train_loss = running.item() / len(train_loader.dataset)
        train_acc = correct.item() / len(train_loader.dataset)
        val_loss, val_acc = evaluate_loader(model, val_loader, ce)

        hist["train_loss"].append(train_loss); hist["train_acc"].append(train_acc)
        hist["val_loss"].append(val_loss);     hist["val_acc"].append(val_acc)

        mark = ""
        if val_acc > best_acc:   # keep the best epoch
            best_acc, best_wts, mark = val_acc, copy.deepcopy(model.state_dict()), "  *best*"
        print(f"Epoch {ep:2d} | train={train_loss:.4f} acc={train_acc:.4f} | "
              f"val={val_loss:.4f} acc={val_acc:.4f}{mark}")

    print(f"Training time: {(time.time() - start) / 60:.1f} min | best val acc: {best_acc:.4f}")
    model.load_state_dict(best_wts)
    return model, hist


def fit_or_load(file, arch, loaders, **train_kw):
    # Train a new model and save it, or load it from SAVE_DIR if it has already been trained
    w_path = os.path.join(SAVE_DIR, file + ".pth")
    h_path = os.path.join(SAVE_DIR, file + "_history.json")
    model = make_model(arch)
    if os.path.exists(w_path) and os.path.exists(h_path) and not FORCE_RETRAIN:
        model.load_state_dict(torch.load(w_path, map_location=device, weights_only=True))
        with open(h_path) as fh:
            hist = json.load(fh)
        print(f"Loaded {w_path} (best val acc {max(hist['val_acc']):.4f}) — set FORCE_RETRAIN = True to train again")
    else:
        model, hist = train_epochs(model, loaders["train"], loaders["val"], **train_kw)
        torch.save(model.state_dict(), w_path)
        with open(h_path, "w") as fh:
            json.dump(hist, fh)
    return model.eval(), hist


def plot_curves(hist, title=""):
    # Loss and accuracy curves for training and validation
    epochs = range(1, len(hist["train_loss"]) + 1)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.5))
    ax1.plot(epochs, hist["train_loss"], label="train_loss")
    ax1.plot(epochs, hist["val_loss"], label="val_loss")
    ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss"); ax1.set_title(title + " loss")
    ax1.legend(); ax1.grid(True, alpha=0.3)
    ax2.plot(epochs, hist["train_acc"], label="train_acc")
    ax2.plot(epochs, hist["val_acc"], label="val_acc")
    ax2.set_xlabel("Epoch"); ax2.set_ylabel("Accuracy"); ax2.set_ylim(0, 1); ax2.set_title(title + " accuracy")
    ax2.legend(); ax2.grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()

In [ ]:
# ---------------- Evaluation helpers ----------------

def get_predictions(model, loader):
    # True labels, predicted labels and softmax probabilities (the probabilities are needed for n-best)
    model.eval(); ys, probs = [], []
    with torch.inference_mode():   # full fp32 precision for the final test results
        for xb, yb in loader:
            out = model(xb.to(device, non_blocking=True, memory_format=torch.channels_last))
            probs.append(torch.softmax(out.float(), dim=1).cpu())
            ys.append(yb)
    probs = torch.cat(probs).numpy()
    y_true = torch.cat(ys).numpy()
    return y_true, probs.argmax(1), probs


results = {}   # test results of every model, used for the comparisons

def test_model(model, name, loader):
    # Accuracy + macro precision / recall / F1 on the test set, stored in `results`
    y_true, y_pred, probs = get_predictions(model, loader)
    acc = accuracy_score(y_true, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
    results[name] = dict(y_true=y_true, y_pred=y_pred, probs=probs, paths=list(loader.dataset.paths),
                         accuracy=acc, precision=prec, recall=rec, f1=f1)
    print(f"{name:22s} | accuracy {acc:.4f} | precision {prec:.4f} | recall {rec:.4f} | F1 {f1:.4f}")


def show_confusion_matrices(name, labels=class_names, min_pct_annot=0.05, hide_diag_on_norm=False):
    # Raw counts and row-normalized confusion matrix side by side (made bigger for 23 classes)
    y_true, y_pred = results[name]["y_true"], results[name]["y_pred"]
    n = len(labels)
    cm_raw  = confusion_matrix(y_true, y_pred, labels=range(n))
    cm_norm = confusion_matrix(y_true, y_pred, labels=range(n), normalize="true")

    fig, axes = plt.subplots(1, 2, figsize=(24, 10))
    ann_raw = np.where(cm_raw == 0, "", cm_raw.astype(int))   # hide zeros
    sns.heatmap(cm_raw, ax=axes[0], annot=ann_raw, fmt="", cmap="Blues", cbar=False, square=True,
                linewidths=.5, linecolor="white", annot_kws={"size": 7},
                xticklabels=labels, yticklabels=labels)
    axes[0].set_title(f"{name} — confusion matrix (counts)")

    ann_norm = np.where(cm_norm >= min_pct_annot, np.round(cm_norm * 100).astype(int).astype(str), "")
    mask = np.eye(n, dtype=bool) if hide_diag_on_norm else None
    sns.heatmap(cm_norm, ax=axes[1], annot=ann_norm, fmt="", cmap="Blues", vmin=0, vmax=1, square=True,
                linewidths=.5, linecolor="white", mask=mask, annot_kws={"size": 7},
                xticklabels=labels, yticklabels=labels)
    diag = " (diagonal hidden)" if hide_diag_on_norm else ""
    axes[1].set_title(f"{name} — normalized % (≥{int(min_pct_annot * 100)} % annotated){diag}")

    for ax in axes:
        ax.set_xlabel("Predicted"); ax.set_ylabel("True")
        ax.tick_params(axis="x", labelrotation=90, labelsize=8)
        ax.tick_params(axis="y", labelrotation=0, labelsize=8)
    plt.tight_layout(); plt.show()


def most_confused(name, top=10):
    # The most common (true -> predicted) mistakes
    cm = confusion_matrix(results[name]["y_true"], results[name]["y_pred"], labels=range(num_classes))
    np.fill_diagonal(cm, 0)
    pairs = sorted([(cm[i, j], class_names[i], class_names[j])
                    for i in range(num_classes) for j in range(num_classes) if cm[i, j] > 0], reverse=True)
    print(f"--- {name}: most common errors (true → predicted) ---")
    for c, t, p in pairs[:top]:
        print(f"{c:3d}   {t}  →  {p}")


def show_misclassified(name, k=20, nrow=5):
    # Grid of k random wrong test predictions: true / predicted species + confidence
    r = results[name]
    wrong = np.where(r["y_true"] != r["y_pred"])[0]
    print(f"{name}: {len(wrong)} wrong out of {len(r['y_true'])} test images")
    pick_ = random.sample(list(wrong), min(k, len(wrong)))
    rows = (len(pick_) + nrow - 1) // nrow
    fig, axs = plt.subplots(rows, nrow, figsize=(16, 3.6 * rows))
    for ax in np.atleast_1d(axs).ravel():
        ax.axis("off")
    for ax, i in zip(np.atleast_1d(axs).ravel(), pick_):
        ax.imshow(Image.open(r["paths"][i]))
        ax.set_title(f"T: {class_names[r['y_true'][i]]}\nP: {class_names[r['y_pred'][i]]} ({r['probs'][i].max():.2f})",
                     fontsize=8, color="red")
    plt.tight_layout(); plt.show()


def top_n_accuracy(probs, y_true, n):
    # Correct if the true class is among the n highest probabilities
    top_n = np.argsort(probs, axis=1)[:, -n:]
    return np.mean([y_true[i] in top_n[i] for i in range(len(y_true))])

In [ ]:
# Sanity check: a batch of training images after augmentation
show_batch(loaders["train"])

## 7. Training round 1 — baseline

**Training settings** (identical for both models → fair comparison)
- Optimizer: Adam, learning rate `1e-4` (small, because the weights are already pretrained), weight decay `1e-4`
- StepLR: learning rate × 0.1 every 7 epochs
- 15 epochs, batch size 32
- The weights of the epoch with the best **validation** accuracy are kept

Two versions of the training data are compared:
- **7a — no augmentation:** training images get the same resize + center crop as validation / test
  (every epoch sees exactly the same images)
- **7b — augmented:** `RandomResizedCrop` + `RandomHorizontalFlip` as in the tutorial (different every epoch)

Validation and test pipelines are identical in both, so the only difference is the augmentation.

### 7a. No augmentation

In [ ]:
EPOCHS = 15

loaders_noaug = make_loaders(val_tfms, splits)
show_batch(loaders_noaug["train"], title="Training batch without augmentation")

In [ ]:
model18_noaug, hist18_noaug = fit_or_load("resnet18_noaug", "resnet18", loaders_noaug,
                                          epochs=EPOCHS, lr=1e-4, wd=1e-4, step_size=7)
plot_curves(hist18_noaug, "ResNet18 no augmentation —")

In [ ]:
model50_noaug, hist50_noaug = fit_or_load("resnet50_noaug", "resnet50", loaders_noaug,
                                          epochs=EPOCHS, lr=1e-4, wd=1e-4, step_size=7)
plot_curves(hist50_noaug, "ResNet50 no augmentation —")

### 7b. Augmented (tutorial recipe)

In [ ]:
model18, hist18 = fit_or_load("resnet18_round1", "resnet18", loaders,
                              epochs=EPOCHS, lr=1e-4, wd=1e-4, step_size=7)
plot_curves(hist18, "ResNet18 augmented —")

In [ ]:
model50, hist50 = fit_or_load("resnet50_round1", "resnet50", loaders,
                              epochs=EPOCHS, lr=1e-4, wd=1e-4, step_size=7)
plot_curves(hist50, "ResNet50 augmented —")

In [ ]:
# Solid = validation, dashed = training -> the gap between them shows overfitting
runs = {"ResNet18": {"no augmentation": hist18_noaug, "augmented": hist18},
        "ResNet50": {"no augmentation": hist50_noaug, "augmented": hist50}}

fig, axs = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
for ax, (arch, hists) in zip(axs, runs.items()):
    for (stage, h), color in zip(hists.items(), ["C0", "C1"]):
        ep = range(1, len(h["val_acc"]) + 1)
        ax.plot(ep, h["val_acc"], color=color, marker="o", label=f"{stage} val")
        ax.plot(ep, h["train_acc"], color=color, ls="--", alpha=0.6, label=f"{stage} train")
    ax.set_title(f"{arch} — train (dashed) vs val (solid)"); ax.set_xlabel("Epoch")
    ax.grid(True, alpha=0.3); ax.legend(fontsize=8)
axs[0].set_ylabel("Accuracy")
plt.tight_layout(); plt.show()

pd.DataFrame({f"{arch} {stage}": {"best val acc (%)": max(h["val_acc"]) * 100,
                                   "best epoch": int(np.argmax(h["val_acc"])) + 1,
                                   "final train acc (%)": h["train_acc"][-1] * 100}
              for arch, hists in runs.items() for stage, h in hists.items()}).T.round(2)

**Training observations**
- **No augmentation:** training accuracy reaches ~100 % by epoch 3–5 while validation accuracy levels off at
  ~92–94 % → the models memorise the training set (overfitting).
- **Augmented:** training accuracy stays at ~92–94 %, close to the validation accuracy → much less overfitting.
  The best epochs come late (13–14 of 15), so longer training could still help a little.
- The StepLR drop after epoch 7 is clearly visible: the curves become smooth and validation accuracy jumps up.
- ResNet50 is better than ResNet18 on validation in both setups (our reference run: no augmentation 93.8 % vs 92.5 %,
  augmented 93.0 % vs 91.9 %).
- Training time for 15 epochs: ResNet18 ~1.5 min and ResNet50 ~3 min on an RTX 4070 laptop GPU;
  roughly 3× longer on a Colab T4.

## 8. Testing on the test set

The test set (472 images) was not used for training or for choosing the best epoch.
Precision, recall and F1 are **macro averages** (computed per species, then averaged).

In [ ]:
test_model(model18_noaug, "ResNet18 no aug", loaders_noaug["test"])
test_model(model50_noaug, "ResNet50 no aug", loaders_noaug["test"])
test_model(model18, "ResNet18", loaders["test"])
test_model(model50, "ResNet50", loaders["test"])

Per-species precision, recall and F1 of the augmented round-1 models:

In [ ]:
for name in ["ResNet18", "ResNet50"]:
    print("=" * 25, name, "=" * 25)
    print(classification_report(results[name]["y_true"], results[name]["y_pred"], labels=range(num_classes),
                                target_names=class_names, digits=3, zero_division=0))

### 8.1 No augmentation vs. augmented training data
Same models, same split, same training settings and the identical test pipeline — the only difference is
augmentation during training.

In [ ]:
compare = pd.DataFrame({
    arch: {"no aug acc (%)":    results[f"{arch} no aug"]["accuracy"] * 100,
           "augmented acc (%)": results[arch]["accuracy"] * 100,
           "no aug F1 (%)":     results[f"{arch} no aug"]["f1"] * 100,
           "augmented F1 (%)":  results[arch]["f1"] * 100}
    for arch in ["ResNet18", "ResNet50"]}).T
compare["acc gain"] = compare["augmented acc (%)"] - compare["no aug acc (%)"]
compare["F1 gain"] = compare["augmented F1 (%)"] - compare["no aug F1 (%)"]
compare.round(2)

**No augmentation vs. augmented:** the difference in test accuracy is small (in our reference run
ResNet18 88.8 % vs 88.1 %, ResNet50 90.9 % vs 90.5 %) — within run-to-run noise (one test image = 0.21 %-points).
Augmentation clearly reduced overfitting (train accuracy ~93 % instead of 100 %), but it did **not** improve
test accuracy within 15 epochs. Section 10 shows one reason: for most photos the horizontal flip is not a mirror
image but an upside-down tree.

### 8.2 Confusion matrices
Rows = true species, columns = predicted species. Everything outside the diagonal is an error.

In [ ]:
show_confusion_matrices("ResNet18")
show_confusion_matrices("ResNet50")

In [ ]:
# Hide the diagonal on the normalized matrix -> the errors stand out
show_confusion_matrices("ResNet50", hide_diag_on_norm=True)

## 9. N-best (top-n) results

A prediction counts as correct if the true species is among the **n most probable** species.
n = 1 is the normal accuracy; when n grows, both models go towards 100 %.

**Task:** find the value of n beyond which the n-best results no longer separate the two models.

The test set has 472 images, so one image ≈ 0.21 %-points. We treat a difference smaller than **1 %-point**
(~5 images) as noise → the models are "not separated" at that n.

In [ ]:
MAX_N = 10
ns = list(range(1, MAX_N + 1))

def nbest_table(name_a, name_b):
    # Top-n accuracy (%) of two models and their difference for n = 1..MAX_N
    t = pd.DataFrame({name: [top_n_accuracy(results[name]["probs"], results[name]["y_true"], n) * 100 for n in ns]
                      for name in [name_a, name_b]}, index=pd.Index(ns, name="n"))
    t["difference"] = t[name_b] - t[name_a]
    return t


def plot_nbest(t, name_a, name_b):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    for name in [name_a, name_b]:
        ax1.plot(ns, t[name], marker="o", label=name)
    ax1.set_xlabel("n"); ax1.set_ylabel("Top-n accuracy (%)"); ax1.set_title("N-best accuracy")
    ax1.set_xticks(ns); ax1.grid(True, alpha=0.3); ax1.legend()
    ax2.bar(ns, t["difference"])
    ax2.axhline(1, color="red", ls="--"); ax2.axhline(-1, color="red", ls="--")
    ax2.set_xlabel("n"); ax2.set_ylabel("%-points"); ax2.set_xticks(ns)
    ax2.set_title(f"{name_b} − {name_a} (red = ±1 %-point)")
    plt.tight_layout(); plt.show()
    separated = [n for n in ns if abs(t.loc[n, "difference"]) >= 1.0]
    print("n values where the models differ by at least 1 %-point:", separated)
    beyond = [n for n in ns if all(abs(t.loc[m, "difference"]) < 1.0 for m in ns if m >= n)]
    if beyond:
        print(f"From n = {beyond[0]} on the difference stays below 1 %-point -> "
              f"n-best stops separating the models beyond n = {beyond[0] - 1}")
    else:
        print(f"The difference is still ≥ 1 %-point at n = {MAX_N} -> no such n within n ≤ {MAX_N}")


nbest_r1 = nbest_table("ResNet18", "ResNet50")
plot_nbest(nbest_r1, "ResNet18", "ResNet50")
nbest_r1.round(2)

**N-best finding.** In our reference run ResNet50 was 2.3 %-points better than ResNet18 at n = 1
(90.5 % vs 88.1 %). The gap shrank to ~1–1.7 %-points for n ≥ 6 but stayed above the 1 %-point threshold up to
n = 10, so within n ≤ 10 there was **no clear n beyond which the models stop separating**; at n ≥ 6 the difference
is only 5–8 images, so the separation there is weak. Both models already have the correct species in their
top 3 for 95–97 % of the test images.

The answer depends on the run: in Tomi's run (own 70/15/15 split, Colab T4) the difference dropped below
1 %-point from **n = 3** on, i.e. n-best stopped separating the models beyond n = 2. The printed result of the
cell above gives the answer for the current run.

## 10. Error analysis

Which species get mixed up, and what do the wrong images look like? We look for **systematic** errors — they
show dataset problems or model weaknesses that a second training round could fix.

In [ ]:
most_confused("ResNet18")
most_confused("ResNet50")

In [ ]:
show_misclassified("ResNet50", k=20)

In [ ]:
# How much of the tree does RandomResizedCrop keep? Default scale = 8–100 % of the image area.
img = Image.open(splits["train"][0][1]).convert("RGB")
fig, axs = plt.subplots(2, 5, figsize=(14, 6))
for j in range(5):
    axs[0, j].imshow(T.RandomResizedCrop(224)(img))
    axs[1, j].imshow(T.RandomResizedCrop(224, scale=(0.5, 1.0))(img))
    for ax in axs[:, j]:
        ax.set_xticks([]); ax.set_yticks([])
axs[0, 0].set_ylabel("scale 0.08–1 (default)"); axs[1, 0].set_ylabel("scale 0.5–1")
plt.suptitle("RandomResizedCrop: default vs. bigger minimum scale"); plt.tight_layout(); plt.show()

### 10.1 Data problems found in the images

While going through the misclassified images we noticed three problems in the dataset itself:

1. **Sideways photos.** Most photos are stored *sideways* — the tree top points to the **left** (the phone was
   held upright, but the file has no EXIF rotation tag, so `exif_transpose` cannot fix it). We checked 96 random
   photos by eye: ~94 % of the landscape (1600 × 1200) photos were sideways, all portrait photos were upright.
   Because of this, `RandomHorizontalFlip` does **not** mirror these trees — it turns them **upside down**.
2. **Truncated photos.** Some JPEG files are cut off: the bottom part of the image is a flat grey area (RGB 128).
3. **Duplicate photos.** Some photos exist twice — also across the train / validation split.

In [ ]:
# 1) Orientation: landscape photos are stored sideways (tree top on the left)
df["landscape"] = [Image.open(p).size[0] > Image.open(p).size[1] for p in df.path]
print(f"Landscape photos: {df.landscape.sum()} / {len(df)} ({df.landscape.mean():.0%})")

ex = df[df.landscape].sample(4, random_state=1).index
fig, axs = plt.subplots(3, 4, figsize=(12, 9))
for j, i in enumerate(ex):
    im = Image.open(df.small[i])
    axs[0, j].imshow(im); axs[0, j].set_title("as stored", fontsize=9)
    axs[1, j].imshow(im.transpose(Image.Transpose.FLIP_LEFT_RIGHT)); axs[1, j].set_title("RandomHorizontalFlip", fontsize=9)
    axs[2, j].imshow(im.transpose(Image.Transpose.ROTATE_270)); axs[2, j].set_title("rotated 90° clockwise", fontsize=9)
for ax in axs.ravel():
    ax.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# 2) Truncated photos: count the flat grey (RGB 128) rows at the bottom of each preprocessed image
def grey_rows_at_bottom(im, tol=3):
    a = np.asarray(im.convert("RGB")).astype(np.int16)
    flat = (np.abs(a - 128) <= tol).all(axis=2).all(axis=1)   # True for rows that are completely grey
    return int(np.argmin(flat[::-1])) if not flat.all() else len(flat)

df["grey_frac"] = [grey_rows_at_bottom(Image.open(p)) / Image.open(p).height for p in df.small]
df["damaged"] = df.grey_frac > 0.01
print("Truncated photos per split:"); print(df[df.damaged].split.value_counts().to_string())
print(f"Badly damaged (> 50 % grey): {(df.grey_frac > 0.5).sum()}")

ex = df[df.damaged].sort_values("grey_frac").index[::max(1, df.damaged.sum() // 6)][:6]
fig, axs = plt.subplots(1, 6, figsize=(16, 3))
for ax, i in zip(axs, ex):
    ax.imshow(Image.open(df.small[i])); ax.set_title(f"{df.grey_frac[i]:.0%} grey", fontsize=9); ax.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# 3) Exact duplicates: identical file content (MD5 hash of the original files)
df["md5"] = [hashlib.md5(open(p, "rb").read()).hexdigest() for p in df.path]
dup_groups = df[df.duplicated("md5", keep=False)].groupby("md5")
print("Duplicate groups:", dup_groups.ngroups, "| photos in them:", sum(len(g) for _, g in dup_groups))
cross = [g for _, g in dup_groups if g.split.nunique() > 1]
print("Groups that cross the split:", len(cross), "->", pd.concat(cross).split.value_counts().to_dict())
print("Duplicates with different labels:", sum(g.species.nunique() > 1 for _, g in dup_groups))

In [ ]:
# Do these problems explain errors? Accuracy of the round-1 ResNet50 on different groups of test images
r = results["ResNet50"]
test_df = df[df.split == "test"].reset_index(drop=True)
assert list(test_df.small) == r["paths"]
test_df["correct"] = r["y_true"] == r["y_pred"]
pd.DataFrame({"images": [len(test_df), (~test_df.landscape).sum(), test_df.landscape.sum(), test_df.damaged.sum()],
              "accuracy (%)": [test_df.correct.mean() * 100,
                               test_df[~test_df.landscape].correct.mean() * 100,
                               test_df[test_df.landscape].correct.mean() * 100,
                               test_df[test_df.damaged].correct.mean() * 100]},
             index=["all test images", "portrait (upright)", "landscape (mostly sideways)", "truncated"]).round(1)

**What we noticed**
- **Similar species are confused by both models**, e.g. *Sapindus saponaria* → *Koelreuteria paniculata*
  (both belong to the soapberry family, Sapindaceae), *Liquidambar formosana* / *Acer palmatum* → *Ginkgo biloba*
  (similar lobed / fan-shaped leaves and yellow autumn colors), *Magnolia liliflora* ↔ *Liriodendron chinense* /
  *Michelia chapensis* (all Magnoliaceae). *Liriodendron chinense* is a frequent wrong prediction in all our runs.
- Visually distinctive species (*Cedrus deodara*, *Salix babylonica*, *Prunus cerasifera* f. *atropurpurea*,
  *Lagerstroemia indica*) are recognised almost perfectly.
- Many wrong images are **winter photos** (bare branches), photos with strong light / shadows, or photos where the
  tree is **small** and the image is full of buildings, cars and road.
- The default `RandomResizedCrop` can keep only 8 % of the image — often just sky, road or a wall, so the model
  learns from crops that contain no tree.
- Wrong predictions often have **low confidence**, and the right species is usually in the top-2 / top-3 (n-best).
- **Data problems:** most photos are stored sideways (horizontal flip turns them upside down), ~2 % of the photos
  are truncated (grey area), and 10 validation photos have an exact copy in the training set, which makes the
  validation accuracy slightly optimistic. The duplicates do not reach the test set.

**Changes for round 2** (the same changes were tried in all three team members' notebooks)
1. `RandomResizedCrop(224, scale=(0.5, 1.0))` → crops always contain a big part of the tree
2. `ColorJitter` → robustness to light and season differences
3. small `RandomRotation(10)` → photos are not always straight
4. **label smoothing 0.1** → less over-confident between similar species
5. **20 epochs** (learning rate step every 8 epochs) because the stronger augmentation makes training harder

## 11. Training round 2 — improved models

Stronger augmentation and label smoothing, based on the error analysis (see the list of changes above).
The validation and test sets and their transforms are unchanged, so the results are directly comparable with
round 1.

In [ ]:
train_tfms_v2 = T.Compose([
    T.RandomResizedCrop(224, scale=(0.5, 1.0)),   # crops always contain a big part of the tree
    T.RandomHorizontalFlip(),
    T.RandomRotation(10),                          # photos are not always straight
    T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.05),   # light / season differences
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

loaders_v2 = make_loaders(train_tfms_v2, splits)   # val / test unchanged -> results are comparable
EPOCHS_V2 = 20

show_batch(loaders_v2["train"], title="Training batch with round-2 augmentation")

### 11.1 ResNet18 v2

In [ ]:
model18_v2, hist18_v2 = fit_or_load("resnet18_round2", "resnet18", loaders_v2, epochs=EPOCHS_V2,
                                    lr=1e-4, wd=1e-4, step_size=8, label_smoothing=0.1)
plot_curves(hist18_v2, "ResNet18 v2 —")

### 11.2 ResNet50 v2

In [ ]:
model50_v2, hist50_v2 = fit_or_load("resnet50_round2", "resnet50", loaders_v2, epochs=EPOCHS_V2,
                                    lr=1e-4, wd=1e-4, step_size=8, label_smoothing=0.1)
plot_curves(hist50_v2, "ResNet50 v2 —")

### 11.3 Test results of the improved models

In [ ]:
test_model(model18_v2, "ResNet18 v2", loaders_v2["test"])
test_model(model50_v2, "ResNet50 v2", loaders_v2["test"])

In [ ]:
show_confusion_matrices("ResNet50 v2", hide_diag_on_norm=True)
most_confused("ResNet50 v2")

In [ ]:
# Round 1 vs round 2 on the same test images, by image group
for name in ["ResNet50", "ResNet50 v2"]:
    test_df[name] = results[name]["y_true"] == results[name]["y_pred"]
groups = {"all test images": test_df.index == test_df.index, "portrait": ~test_df.landscape,
          "landscape": test_df.landscape, "truncated": test_df.damaged}
pd.DataFrame({g: {"images": int(m.sum()),
                  "ResNet50 round 1 (%)": test_df[m]["ResNet50"].mean() * 100,
                  "ResNet50 v2 (%)": test_df[m]["ResNet50 v2"].mean() * 100} for g, m in groups.items()}).T.round(1)

## 12. Final comparison

In [ ]:
summary = pd.DataFrame({name: {k: r[k] * 100 for k in ["accuracy", "precision", "recall", "f1"]}
                        for name, r in results.items()}).T
summary.columns = ["accuracy (%)", "precision (%)", "recall (%)", "F1 (%)"]
summary.round(2)

In [ ]:
# Top-n accuracy of all models
all_names = list(results.keys())
nbest_all = pd.DataFrame({name: [top_n_accuracy(results[name]["probs"], results[name]["y_true"], n) * 100 for n in ns]
                          for name in all_names}, index=pd.Index(ns, name="n"))

plt.figure(figsize=(9, 5))
for name in all_names:
    plt.plot(ns, nbest_all[name], marker="o", label=name)
plt.xlabel("n"); plt.ylabel("Top-n accuracy (%)"); plt.title("N-best accuracy — all models")
plt.xticks(ns); plt.ylim(84, 100.5); plt.grid(True, alpha=0.3); plt.legend(); plt.show()
nbest_all.round(2)

In [ ]:
# N-best separation for the improved (round 2) models
nbest_r2 = nbest_table("ResNet18 v2", "ResNet50 v2")
plot_nbest(nbest_r2, "ResNet18 v2", "ResNet50 v2")
nbest_r2.round(2)